In [ ]:
#@title Cell 1 - Notebook overview

from IPython.display import display, Markdown

display(Markdown(r"""
# Simulation 04: Pathogen-dependent plasmid features

## Purpose

Simulation 4 starts from the finalized Simulation 1 biological model and changes one identifiability condition:

\[
\boxed{\text{some plasmid-associated properties depend on the pathogen}}
\]

The agreed Simulation 4 comparison is:

1. **Realized-feature fit** — the model receives the realized pathogen-specific copy-number and fitness values.
2. **Fixed-feature fit** — the same data are fitted while incorrectly treating copy number and fitness as fixed properties of the plasmid across pathogens.

The true MICs are generated from the pathogen-dependent realized values in both fits.

The central question remains:

\[
\boxed{\text{Does the effect of plasmid }j\text{ depend on the pathogen chromosome }i?}
\]

The model is

\[
y_{ij}
=
\alpha
+
\boldsymbol{\beta}_C^T\mathbf c_i
+
\boldsymbol{\beta}_P^T\mathbf p_{ij}
+
\mathbf c_i^T\mathbf B\mathbf p_{ij}
+
u_i
+
\varepsilon_{ij},
\]

where \(\mathbf p_{ij}\) is allowed to contain pathogen-dependent realized plasmid features.

## What is retained from Simulation 1

- 200 pathogen chromosomal backgrounds.
- 20 blaTEM-1 plasmids plus \(P_0\).
- Complete 200 × 21 observation design.
- The same 30 gene-centred chromosomal units and biological chromosome coefficients.
- The same empirical promoter-genotype/copy-number pairs as the plasmid baseline.
- The same promoter effects.
- The same TEM-1 reference effect.
- The same chromosome–plasmid interaction structure and magnitude.
- 500 background SNPs for \(K\).
- \(\alpha=-2.5\).
- \(\sigma_g=0.40\), \(\sigma_e=0.32\).
- 100 simulation replicates.
- 200 representative bootstrap replicates.

## Simulation 4 extension

The plasmid feature vector is

\[
\mathbf p_{ij}
=
\left(
1,\,
I(C32T),\,
I(G162T),\,
I(G175A),\,
q_{ij},\,
f_{ij}
\right)^T,
\]

where:

- \(q_{ij}\) is the realized log2 copy number for plasmid \(j\) in pathogen \(i\), relative to the empirical median;
- \(f_{ij}\) is a standardized plasmid-associated fitness score realized in pathogen \(i\).

The empirical copy number sampled for plasmid \(j\) is the baseline \(q_j\). A simulated baseline fitness score \(f_j\) is assigned to each plasmid. Pathogen-specific shifts plus smaller pathogen–plasmid deviations produce \(q_{ij}\) and \(f_{ij}\).

The fixed-feature comparison incorrectly substitutes \(q_j\) and \(f_j\) for all pathogens.

This directly tests the identifiability condition that pathogen-dependent plasmid features must be known or explicitly assumed after transformation.
"""))

print("Transition: Cell 2 loads the public plasmid-feature input and defines Simulation 4 settings.")


In [ ]:
#@title Cell 2 - Load public plasmid-feature input and define fixed settings

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd

from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize_scalar

REPO_ROOT = Path.cwd()
DATA_FILE = REPO_ROOT / "data" / "plasmid_feature_pairs.csv"

OUTPUT_DIR = (
    REPO_ROOT
    / "results"
    / "simulation_04_pathogen_dependent_properties"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Public plasmid-feature input was not found:\n"
        f"{DATA_FILE}\n"
        "Run the notebook from the repository root."
    )

empirical_pairs = pd.read_csv(
    DATA_FILE,
    low_memory=False,
)

KEY_SITE_COLUMNS = [
    "sutcliffe_32_nt",
    "sutcliffe_162_nt",
    "sutcliffe_175_nt",
]

required_columns = [
    *KEY_SITE_COLUMNS,
    "CN_TEM1",
]

missing_columns = [
    c for c in required_columns
    if c not in empirical_pairs.columns
]

if missing_columns:
    raise RuntimeError(
        "The public plasmid-feature input is missing required columns:\n"
        + "\n".join(f"  - {c}" for c in missing_columns)
    )

empirical_pairs = empirical_pairs[
    required_columns
].copy()

for col in KEY_SITE_COLUMNS:
    empirical_pairs[col] = (
        empirical_pairs[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

empirical_pairs["CN_TEM1"] = pd.to_numeric(
    empirical_pairs["CN_TEM1"],
    errors="coerce",
)

valid_nt = {"A", "C", "G", "T"}

complete_mask = (
    empirical_pairs[KEY_SITE_COLUMNS]
    .apply(lambda s: s.isin(valid_nt))
    .all(axis=1)
    & empirical_pairs["CN_TEM1"].notna()
    & (empirical_pairs["CN_TEM1"] > 0)
)

empirical_pairs = (
    empirical_pairs.loc[complete_mask]
    .reset_index(drop=True)
)

if len(empirical_pairs) < 20:
    raise RuntimeError(
        f"Only {len(empirical_pairs)} complete promoter/copy-number pairs remain; "
        "at least 20 are required."
    )

CN_REFERENCE = float(
    empirical_pairs["CN_TEM1"].median()
)

if not np.isfinite(CN_REFERENCE) or CN_REFERENCE <= 0:
    raise RuntimeError(
        "The median CN_TEM1 is not positive."
    )

# -------------------------------------------------------------------------
# Simulation 1 biological settings: unchanged unless explicitly noted below
# -------------------------------------------------------------------------

MASTER_SEED = 20260906

N_PATHOGENS = 200
N_PLASMIDS = 20

PREDEFINED_GENES = [
    "acrB", "acrR", "ampC", "basR", "cirA", "cyaA", "fabI", "folP", "ftsI",
    "gyrA", "marR", "nfsA", "nfsB", "ompC", "ompF", "parC", "parE", "pmrB",
    "ptsI", "rpoB", "rpsL", "soxR", "soxS", "uhpT", "acrA", "tolC", "marA",
    "rob", "ompR", "envZ",
]

TARGET_FEATURE_LABELS = []
for gene in PREDEFINED_GENES:
    TARGET_FEATURE_LABELS.extend([
        f"{gene}:coding",
        f"{gene}:upstream_300bp",
    ])

D_C = len(TARGET_FEATURE_LABELS)

PLASMID_FEATURE_LABELS = [
    "TEM1_plasmid_presence",
    "C32T",
    "G162T",
    "G175A",
    "log2_CN_relative_to_empirical_median",
    "pathogen_dependent_fitness_score",
]
D_P = len(PLASMID_FEATURE_LABELS)

CHROMOSOMAL_STATES = np.array([-1.0, 0.0, 1.0])
CHROMOSOMAL_STATE_PROBS = np.array([0.15, 0.70, 0.15])

N_BACKGROUND_SNPS = 500
ALLELE_FREQ_LOW = 0.10
ALLELE_FREQ_HIGH = 0.90

ALPHA_TRUE = -2.5

SIGMA_G_TRUE = 0.40
SIGMA_E_TRUE = 0.32
SIGMA_G2_TRUE = SIGMA_G_TRUE ** 2
SIGMA_E2_TRUE = SIGMA_E_TRUE ** 2

# Simulation 1 coefficients plus one Scenario 4 fitness coefficient.
BETA_P_TRUE = np.array([
    0.50,   # TEM-1 plasmid presence
    0.50,   # C32T
    0.25,   # G162T
    0.00,   # G175A
    0.25,   # log2 copy number relative to empirical median
    0.20,   # standardized plasmid-associated fitness score
], dtype=float)

INTERACTION_MAGNITUDE = 0.10

# -------------------------------------------------------------------------
# Simulation 4 pathogen-dependent feature settings
# -------------------------------------------------------------------------

# These are simulation assumptions, not empirical constants.
# Values are on the feature scales used in the fitted model.
BASELINE_FITNESS_SD = 0.50

CN_HOST_SHIFT_SD = 0.35
CN_PAIR_DEVIATION_SD = 0.20

FITNESS_HOST_SHIFT_SD = 0.35
FITNESS_PAIR_DEVIATION_SD = 0.20

N_SIM_REPLICATES = 100
N_BOOTSTRAP = 200

RUN_FULL_BOOTSTRAP_COVERAGE = False

print("=" * 90)
print("SIMULATION 04 — FIXED SETTINGS")
print("=" * 90)
print(f"Empirical promoter/CN pairs available: {len(empirical_pairs):,}")
print(f"Empirical median CN_TEM1:               {CN_REFERENCE:.6f}")
print(f"Pathogens:                              {N_PATHOGENS}")
print(f"Plasmids:                               {N_PLASMIDS}")
print(f"Complete observations:                  {N_PATHOGENS * (N_PLASMIDS + 1):,}")
print(f"Plasmid feature dimension:              {D_P}")
print(f"CN host-shift SD:                       {CN_HOST_SHIFT_SD:.2f}")
print(f"CN pair-deviation SD:                   {CN_PAIR_DEVIATION_SD:.2f}")
print(f"Fitness host-shift SD:                  {FITNESS_HOST_SHIFT_SD:.2f}")
print(f"Fitness pair-deviation SD:              {FITNESS_PAIR_DEVIATION_SD:.2f}")
print(f"Fitness coefficient:                    {BETA_P_TRUE[5]:.2f}")
print(f"alpha:                                  {ALPHA_TRUE}")
print(f"sigma_g:                                {SIGMA_G_TRUE}")
print(f"sigma_e:                                {SIGMA_E_TRUE}")
print(f"Output directory:                       {OUTPUT_DIR}")
print("\nCell 2: PASS")


In [ ]:
#@title Cell 3 - Define Simulation 1 biology and pathogen-dependent plasmid features

def feature_index(gene, region):
    label = (
        f"{gene}:coding"
        if region == "coding"
        else f"{gene}:upstream_300bp"
    )
    return TARGET_FEATURE_LABELS.index(label)


def build_true_chromosomal_coefficients():
    beta_C = np.zeros(D_C, dtype=float)
    B = np.zeros((D_C, D_P), dtype=float)

    efflux_machinery = {"acrA", "acrB", "tolC"}
    repressors = {"acrR", "marR"}
    activators = {"marA", "rob", "soxR", "soxS"}
    porins = {"ompC", "ompF"}

    for gene in PREDEFINED_GENES:
        coding_i = feature_index(gene, "coding")
        upstream_i = feature_index(gene, "upstream")

        if gene in efflux_machinery:
            beta_C[coding_i] = +0.25
            beta_C[upstream_i] = +0.25

        elif gene in repressors:
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.25

        elif gene in activators:
            beta_C[coding_i] = +0.25
            beta_C[upstream_i] = +0.25

        elif gene in porins:
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.25

        elif gene == "ampC":
            beta_C[coding_i] = +0.10
            beta_C[upstream_i] = +0.25

        elif gene == "ftsI":
            beta_C[coding_i] = -0.25
            beta_C[upstream_i] = -0.10

        if gene in efflux_machinery:
            B[coding_i, 0] = +INTERACTION_MAGNITUDE
            B[upstream_i, 0] = +INTERACTION_MAGNITUDE

        elif gene in repressors:
            B[coding_i, 0] = -INTERACTION_MAGNITUDE
            B[upstream_i, 0] = -INTERACTION_MAGNITUDE

        elif gene in activators:
            B[coding_i, 0] = +INTERACTION_MAGNITUDE
            B[upstream_i, 0] = +INTERACTION_MAGNITUDE

        elif gene in porins:
            B[coding_i, 0] = -INTERACTION_MAGNITUDE
            B[upstream_i, 0] = -INTERACTION_MAGNITUDE

    return beta_C, B


BETA_C_TRUE, B_TRUE = build_true_chromosomal_coefficients()


def simulate_targeted_chromosomal_features(rng):
    for _ in range(100):
        C = rng.choice(
            CHROMOSOMAL_STATES,
            size=(N_PATHOGENS, D_C),
            p=CHROMOSOMAL_STATE_PROBS,
        ).astype(float)

        augmented = np.column_stack([
            np.ones(N_PATHOGENS, dtype=float),
            C,
        ])

        if np.linalg.matrix_rank(augmented) == D_C + 1:
            return C

    raise RuntimeError(
        "Could not generate a full-rank pathogen chromosome matrix."
    )


def sample_empirical_plasmids(rng):
    n_empirical = len(empirical_pairs)

    for _ in range(5000):
        selected_positions = rng.choice(
            n_empirical,
            size=N_PLASMIDS,
            replace=False,
        )

        sampled = (
            empirical_pairs.iloc[selected_positions]
            .copy()
            .reset_index(drop=True)
        )

        sampled.insert(
            0,
            "plasmid_id",
            [f"P{j}" for j in range(1, N_PLASMIDS + 1)],
        )

        sampled["I_C32T"] = (
            sampled["sutcliffe_32_nt"].eq("T")
        ).astype(float)

        sampled["I_G162T"] = (
            sampled["sutcliffe_162_nt"].eq("T")
        ).astype(float)

        sampled["I_G175A"] = (
            sampled["sutcliffe_175_nt"].eq("A")
        ).astype(float)

        sampled["q_CN_baseline"] = np.log2(
            sampled["CN_TEM1"].astype(float)
            / CN_REFERENCE
        )

        # Scenario 4 adds one simulated plasmid-level baseline fitness score.
        sampled["fitness_baseline"] = rng.normal(
            0.0,
            BASELINE_FITNESS_SD,
            size=N_PLASMIDS,
        )

        P_fixed = np.column_stack([
            np.ones(N_PLASMIDS, dtype=float),
            sampled["I_C32T"].to_numpy(dtype=float),
            sampled["I_G162T"].to_numpy(dtype=float),
            sampled["I_G175A"].to_numpy(dtype=float),
            sampled["q_CN_baseline"].to_numpy(dtype=float),
            sampled["fitness_baseline"].to_numpy(dtype=float),
        ])

        # Need all six plasmid dimensions to be estimable.
        P0 = np.zeros((1, D_P), dtype=float)
        augmented_state_matrix = np.column_stack([
            np.ones(N_PLASMIDS + 1, dtype=float),
            np.vstack([P0, P_fixed]),
        ])

        if np.linalg.matrix_rank(augmented_state_matrix) == D_P + 1:
            return P_fixed, sampled

    raise RuntimeError(
        "Could not sample 20 full-rank empirical/simulated plasmid profiles."
    )


def simulate_pathogen_dependent_plasmid_features(rng, P_fixed):
    """
    Produce realized p_ij values.

    Presence and promoter features remain plasmid properties.
    Copy number and fitness vary with pathogen.

    q_ij = q_j + host_shift_i + pair_deviation_ij
    f_ij = f_j + host_shift_i + pair_deviation_ij
    """
    P_realized = np.broadcast_to(
        P_fixed[None, :, :],
        (N_PATHOGENS, N_PLASMIDS, D_P),
    ).copy()

    cn_host_shift = rng.normal(
        0.0,
        CN_HOST_SHIFT_SD,
        size=N_PATHOGENS,
    )

    cn_pair_deviation = rng.normal(
        0.0,
        CN_PAIR_DEVIATION_SD,
        size=(N_PATHOGENS, N_PLASMIDS),
    )

    fitness_host_shift = rng.normal(
        0.0,
        FITNESS_HOST_SHIFT_SD,
        size=N_PATHOGENS,
    )

    fitness_pair_deviation = rng.normal(
        0.0,
        FITNESS_PAIR_DEVIATION_SD,
        size=(N_PATHOGENS, N_PLASMIDS),
    )

    P_realized[:, :, 4] = (
        P_fixed[None, :, 4]
        + cn_host_shift[:, None]
        + cn_pair_deviation
    )

    P_realized[:, :, 5] = (
        P_fixed[None, :, 5]
        + fitness_host_shift[:, None]
        + fitness_pair_deviation
    )

    return {
        "P_realized": P_realized,
        "cn_host_shift": cn_host_shift,
        "cn_pair_deviation": cn_pair_deviation,
        "fitness_host_shift": fitness_host_shift,
        "fitness_pair_deviation": fitness_pair_deviation,
    }


def simulate_background_relatedness(rng):
    source_frequencies = rng.uniform(
        ALLELE_FREQ_LOW,
        ALLELE_FREQ_HIGH,
        size=N_BACKGROUND_SNPS,
    )

    G = rng.binomial(
        1,
        source_frequencies,
        size=(N_PATHOGENS, N_BACKGROUND_SNPS),
    ).astype(float)

    p = G.mean(axis=0)
    Z = G - p[None, :]

    denominator = float(
        np.sum(p * (1.0 - p))
    )

    if denominator <= 0:
        raise ValueError(
            "Background-SNP relatedness denominator is not positive."
        )

    K = (Z @ Z.T) / denominator
    K = (K + K.T) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(K)

    if eigenvalues.min() < -1e-8:
        raise ValueError(
            f"Constructed K is unexpectedly non-PSD: "
            f"minimum eigenvalue={eigenvalues.min():.6g}"
        )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    K = (
        eigenvectors * eigenvalues
    ) @ eigenvectors.T

    K = (K + K.T) / 2.0

    return K, G, p


def build_design_from_realized_features(C, P_realized):
    """
    Full 200 x 21 design using the realized p_ij for P1...P20.
    P0 remains the all-zero plasmid vector.
    """
    rows_C = []
    rows_P = []
    pathogen_index = []
    plasmid_state_index = []

    zero_p = np.zeros(D_P, dtype=float)

    for i in range(N_PATHOGENS):
        rows_C.append(C[i])
        rows_P.append(zero_p)
        pathogen_index.append(i)
        plasmid_state_index.append(0)

        for j in range(N_PLASMIDS):
            rows_C.append(C[i])
            rows_P.append(P_realized[i, j])
            pathogen_index.append(i)
            plasmid_state_index.append(j + 1)

    C_obs = np.asarray(rows_C, dtype=float)
    P_obs = np.asarray(rows_P, dtype=float)
    pathogen_index = np.asarray(pathogen_index, dtype=int)
    plasmid_state_index = np.asarray(plasmid_state_index, dtype=int)

    interaction = np.einsum(
        "ni,nj->nij",
        C_obs,
        P_obs,
    ).reshape(
        len(pathogen_index),
        D_C * D_P,
    )

    X = np.column_stack([
        np.ones(len(pathogen_index), dtype=float),
        C_obs,
        P_obs,
        interaction,
    ])

    return X, pathogen_index, plasmid_state_index


def build_design_from_fixed_plasmid_features(C, P_fixed):
    """
    Same 200 x 21 observations, but incorrectly treats CN and fitness as
    fixed properties of plasmid j across all pathogens.
    """
    P_all = np.vstack([
        np.zeros((1, D_P), dtype=float),
        P_fixed,
    ])

    n_states = N_PLASMIDS + 1

    pathogen_index = np.repeat(
        np.arange(N_PATHOGENS, dtype=int),
        n_states,
    )

    plasmid_state_index = np.tile(
        np.arange(n_states, dtype=int),
        N_PATHOGENS,
    )

    C_obs = C[pathogen_index, :]
    P_obs = P_all[plasmid_state_index, :]

    interaction = np.einsum(
        "ni,nj->nij",
        C_obs,
        P_obs,
    ).reshape(
        len(pathogen_index),
        D_C * D_P,
    )

    X = np.column_stack([
        np.ones(len(pathogen_index), dtype=float),
        C_obs,
        P_obs,
        interaction,
    ])

    return X, pathogen_index, plasmid_state_index


def draw_correlated_host_effect(rng, K, sigma_g2):
    eigenvalues, eigenvectors = np.linalg.eigh(
        (K + K.T) / 2.0
    )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None,
    )

    z = rng.normal(
        0.0,
        1.0,
        size=N_PATHOGENS,
    )

    u = eigenvectors @ (
        np.sqrt(
            sigma_g2 * eigenvalues
        ) * z
    )

    return u


def simulate_scenario4_dataset(seed):
    rng = np.random.default_rng(seed)

    C = simulate_targeted_chromosomal_features(rng)
    P_fixed, sampled_plasmids = sample_empirical_plasmids(rng)

    realized_info = simulate_pathogen_dependent_plasmid_features(
        rng,
        P_fixed,
    )

    P_realized = realized_info["P_realized"]

    X_realized, pathogen_index, plasmid_state_index = (
        build_design_from_realized_features(
            C,
            P_realized,
        )
    )

    X_fixed, pathogen_index_fixed, plasmid_state_index_fixed = (
        build_design_from_fixed_plasmid_features(
            C,
            P_fixed,
        )
    )

    if not np.array_equal(
        pathogen_index,
        pathogen_index_fixed,
    ):
        raise RuntimeError(
            "Realized and fixed designs have inconsistent pathogen row order."
        )

    if not np.array_equal(
        plasmid_state_index,
        plasmid_state_index_fixed,
    ):
        raise RuntimeError(
            "Realized and fixed designs have inconsistent plasmid row order."
        )

    if np.linalg.matrix_rank(X_realized) != X_realized.shape[1]:
        raise RuntimeError(
            "Realized-feature design is rank deficient."
        )

    if np.linalg.matrix_rank(X_fixed) != X_fixed.shape[1]:
        raise RuntimeError(
            "Fixed-feature comparison design is rank deficient."
        )

    K, G_background, background_frequencies = (
        simulate_background_relatedness(rng)
    )

    theta_true = np.concatenate([
        np.array([ALPHA_TRUE], dtype=float),
        BETA_C_TRUE,
        BETA_P_TRUE,
        B_TRUE.reshape(-1),
    ])

    structural_mean = X_realized @ theta_true

    u = draw_correlated_host_effect(
        rng,
        K,
        SIGMA_G2_TRUE,
    )

    epsilon = rng.normal(
        0.0,
        SIGMA_E_TRUE,
        size=X_realized.shape[0],
    )

    y = (
        structural_mean
        + u[pathogen_index]
        + epsilon
    )

    return {
        "seed": int(seed),
        "C": C,
        "P_fixed": P_fixed,
        "P_realized": P_realized,
        "sampled_plasmids": sampled_plasmids,
        "K": K,
        "G_background": G_background,
        "background_frequencies": background_frequencies,
        "beta_C_true": BETA_C_TRUE.copy(),
        "beta_P_true": BETA_P_TRUE.copy(),
        "B_true": B_TRUE.copy(),
        "theta_true": theta_true,
        "structural_mean": structural_mean,
        "u_true": u,
        "epsilon_true": epsilon,
        "X_realized": X_realized,
        "X_fixed": X_fixed,
        "pathogen_index": pathogen_index,
        "plasmid_state_index": plasmid_state_index,
        "y": y,
        **realized_info,
    }


print("Cell 3: PASS")


In [ ]:
#@title Cell 4 - Define efficient REML and GLS fitting

def prepare_reml_static(X, pathogen_index, K):
    X = np.asarray(X, dtype=float)
    K = np.asarray(K, dtype=float)

    m, p_fixed = X.shape

    design_rank = np.linalg.matrix_rank(X)

    if design_rank != p_fixed:
        raise ValueError(
            f"Fixed-effect design matrix is rank deficient: "
            f"rank={design_rank}, columns={p_fixed}."
        )

    eigenvalues, Q = np.linalg.eigh(
        (K + K.T) / 2.0
    )

    keep = eigenvalues > 1e-10
    eigenvalues = eigenvalues[keep]
    Q = Q[:, keep]

    B_lowrank = (
        Q[pathogen_index, :]
        * np.sqrt(eigenvalues)[None, :]
    )

    static = {
        "m": int(m),
        "p_fixed": int(p_fixed),
        "rank_K": int(len(eigenvalues)),
        "B_lowrank": B_lowrank,
        "BtB": B_lowrank.T @ B_lowrank,
        "BtX": B_lowrank.T @ X,
        "XTX": X.T @ X,
        "X": X,
    }

    return static


def add_y_to_reml_static(static, y):
    working = {
        key: value
        for key, value in static.items()
        if key not in {"B_lowrank", "X"}
    }

    B_lowrank = static["B_lowrank"]
    X = static["X"]

    working["Bty"] = B_lowrank.T @ y
    working["Xty"] = X.T @ y
    working["yty"] = float(y @ y)

    return working


def evaluate_profile_reml(log_delta, working, return_fit=False):
    delta = float(np.exp(log_delta))

    m = working["m"]
    p_fixed = working["p_fixed"]
    rank_K = working["rank_K"]

    M = (
        np.eye(rank_K)
        + working["BtB"] / delta
    )

    try:
        chol_M = cho_factor(
            M,
            lower=True,
            check_finite=False,
        )
    except np.linalg.LinAlgError:
        return np.inf if not return_fit else None

    M_inv_BtX = cho_solve(
        chol_M,
        working["BtX"],
        check_finite=False,
    )

    M_inv_Bty = cho_solve(
        chol_M,
        working["Bty"],
        check_finite=False,
    )

    XtAinvX = (
        working["XTX"] / delta
        - (
            working["BtX"].T
            @ M_inv_BtX
        ) / (delta ** 2)
    )

    XtAinvy = (
        working["Xty"] / delta
        - (
            working["BtX"].T
            @ M_inv_Bty
        ) / (delta ** 2)
    )

    yAinvy = (
        working["yty"] / delta
        - float(
            working["Bty"].T
            @ M_inv_Bty
        ) / (delta ** 2)
    )

    XtAinvX = (
        XtAinvX + XtAinvX.T
    ) / 2.0

    sign_X, logdet_X = np.linalg.slogdet(
        XtAinvX
    )

    if sign_X <= 0:
        return np.inf if not return_fit else None

    try:
        chol_X = cho_factor(
            XtAinvX,
            lower=True,
            check_finite=False,
        )

        beta_hat = cho_solve(
            chol_X,
            XtAinvy,
            check_finite=False,
        )
    except np.linalg.LinAlgError:
        return np.inf if not return_fit else None

    q = float(
        yAinvy
        - beta_hat @ XtAinvy
    )

    df_reml = m - p_fixed

    if q <= 0 or df_reml <= 0:
        return np.inf if not return_fit else None

    logdet_M = 2.0 * np.sum(
        np.log(np.diag(chol_M[0]))
    )

    logdet_A = (
        m * np.log(delta)
        + logdet_M
    )

    objective = (
        logdet_A
        + logdet_X
        + df_reml * np.log(q / df_reml)
    )

    if return_fit:
        return {
            "objective": float(objective),
            "beta_hat": beta_hat,
            "q": q,
            "delta": delta,
            "df_reml": int(df_reml),
        }

    return float(objective)


def fit_section2_reml_gls(X, y, pathogen_index, K, static=None):
    if static is None:
        static = prepare_reml_static(
            X,
            pathogen_index,
            K,
        )

    working = add_y_to_reml_static(
        static,
        y,
    )

    optimization = minimize_scalar(
        lambda log_delta: evaluate_profile_reml(
            log_delta,
            working,
            return_fit=False,
        ),
        bounds=(-8.0, 8.0),
        method="bounded",
        options={
            "xatol": 1e-4,
            "maxiter": 100,
        },
    )

    if not optimization.success:
        raise RuntimeError(
            "REML optimization failed: "
            + str(optimization.message)
        )

    fit = evaluate_profile_reml(
        optimization.x,
        working,
        return_fit=True,
    )

    if fit is None:
        raise RuntimeError(
            "Final REML/GLS evaluation failed."
        )

    sigma_g2_hat = (
        fit["q"]
        / fit["df_reml"]
    )

    sigma_e2_hat = (
        fit["delta"]
        * sigma_g2_hat
    )

    return {
        "beta_hat": fit["beta_hat"],
        "sigma_g2_hat": float(sigma_g2_hat),
        "sigma_e2_hat": float(sigma_e2_hat),
        "delta_hat": float(fit["delta"]),
        "reml_objective": float(fit["objective"]),
        "optimization_nfev": int(optimization.nfev),
        "static": static,
    }


def unpack_beta(beta_hat):
    start_C = 1
    stop_C = start_C + D_C

    start_P = stop_C
    stop_P = start_P + D_P

    start_B = stop_P

    alpha_hat = float(beta_hat[0])
    beta_C_hat = beta_hat[start_C:stop_C]
    beta_P_hat = beta_hat[start_P:stop_P]
    B_hat = beta_hat[start_B:].reshape(
        D_C,
        D_P,
    )

    return (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    )


print("Cell 4: PASS")


In [ ]:
#@title Cell 5 - Generate and QC one Scenario 4 dataset

EXAMPLE_SEED = MASTER_SEED

example = simulate_scenario4_dataset(
    EXAMPLE_SEED
)

n_observations = len(example["y"])

q_fixed = example["P_fixed"][:, 4]
q_realized = example["P_realized"][:, :, 4]

f_fixed = example["P_fixed"][:, 5]
f_realized = example["P_realized"][:, :, 5]

feature_qc = pd.DataFrame({
    "feature": [
        "log2_CN_relative_to_empirical_median",
        "pathogen_dependent_fitness_score",
    ],
    "fixed_plasmid_SD": [
        float(np.std(q_fixed, ddof=1)),
        float(np.std(f_fixed, ddof=1)),
    ],
    "realized_pair_SD": [
        float(np.std(q_realized, ddof=1)),
        float(np.std(f_realized, ddof=1)),
    ],
    "realized_minus_fixed_SD": [
        float(np.std(
            q_realized - q_fixed[None, :],
            ddof=1,
        )),
        float(np.std(
            f_realized - f_fixed[None, :],
            ddof=1,
        )),
    ],
})

REALIZED_FEATURE_PATH = (
    OUTPUT_DIR
    / "04_example_realized_pathogen_plasmid_features.csv"
)

rows = []
for i in range(N_PATHOGENS):
    for j in range(N_PLASMIDS):
        rows.append({
            "pathogen_id": f"C{i + 1}",
            "plasmid_id": f"P{j + 1}",
            "q_CN_fixed": q_fixed[j],
            "q_CN_realized": q_realized[i, j],
            "fitness_fixed": f_fixed[j],
            "fitness_realized": f_realized[i, j],
        })

realized_feature_table = pd.DataFrame(rows)
realized_feature_table.to_csv(
    REALIZED_FEATURE_PATH,
    index=False,
)

print("=" * 90)
print("CELL 5 — SIMULATION 4 PATHOGEN-DEPENDENT FEATURE QC")
print("=" * 90)
print(f"Observations:                         {n_observations:,}")
print(f"Realized design dimensions:          {example['X_realized'].shape}")
print(f"Realized design rank:                {np.linalg.matrix_rank(example['X_realized'])}")
print(f"Fixed-feature design dimensions:     {example['X_fixed'].shape}")
print(f"Fixed-feature design rank:           {np.linalg.matrix_rank(example['X_fixed'])}")
print(f"K dimensions:                       {example['K'].shape}")
print(f"Observed log2 MIC mean:              {np.mean(example['y']):.6f}")
print(f"Observed log2 MIC SD:                {np.std(example['y'], ddof=1):.6f}")

print("\nPathogen-dependent feature variation:")
display(feature_qc)

print("\nFirst five realized pathogen–plasmid feature rows:")
display(realized_feature_table.head())

print("\nSaved:")
print(REALIZED_FEATURE_PATH)

print("\nCell 5: PASS")


In [ ]:
#@title Cell 6 - Fit realized-feature and incorrectly fixed-feature models

start_time = time.time()

static_realized = prepare_reml_static(
    example["X_realized"],
    example["pathogen_index"],
    example["K"],
)

fit_realized = fit_section2_reml_gls(
    example["X_realized"],
    example["y"],
    example["pathogen_index"],
    example["K"],
    static=static_realized,
)

time_realized = time.time() - start_time

start_time = time.time()

static_fixed = prepare_reml_static(
    example["X_fixed"],
    example["pathogen_index"],
    example["K"],
)

fit_fixed = fit_section2_reml_gls(
    example["X_fixed"],
    example["y"],
    example["pathogen_index"],
    example["K"],
    static=static_fixed,
)

time_fixed = time.time() - start_time

(
    alpha_realized,
    beta_C_realized,
    beta_P_realized,
    B_realized,
) = unpack_beta(
    fit_realized["beta_hat"]
)

(
    alpha_fixed,
    beta_C_fixed,
    beta_P_fixed,
    B_fixed,
) = unpack_beta(
    fit_fixed["beta_hat"]
)

fit_summary = pd.DataFrame([
    {
        "fit": "realized_pathogen_specific_features",
        "alpha_hat": alpha_realized,
        "sigma_g2_hat": fit_realized["sigma_g2_hat"],
        "sigma_e2_hat": fit_realized["sigma_e2_hat"],
        "variance_ratio_hat": fit_realized["delta_hat"],
        "fit_time_seconds": time_realized,
    },
    {
        "fit": "incorrect_fixed_plasmid_features",
        "alpha_hat": alpha_fixed,
        "sigma_g2_hat": fit_fixed["sigma_g2_hat"],
        "sigma_e2_hat": fit_fixed["sigma_e2_hat"],
        "variance_ratio_hat": fit_fixed["delta_hat"],
        "fit_time_seconds": time_fixed,
    },
])

beta_comparison = pd.DataFrame({
    "feature": PLASMID_FEATURE_LABELS,
    "true_beta_P": BETA_P_TRUE,
    "realized_feature_fit": beta_P_realized,
    "fixed_feature_fit": beta_P_fixed,
})

print("=" * 90)
print("CELL 6 — REALIZED VS FIXED FEATURE FITS")
print("=" * 90)
print(f"True alpha:       {ALPHA_TRUE:.6f}")
print(f"True sigma_g^2:   {SIGMA_G2_TRUE:.6f}")
print(f"True sigma_e^2:   {SIGMA_E2_TRUE:.6f}")

print("\nFit comparison:")
display(fit_summary)

print("\nPlasmid main-effect recovery:")
display(beta_comparison)

print("\nCell 6: PASS")


In [ ]:
#@title Cell 7 - Compare Delta and DeltaDelta recovery between the two fits

def true_effects_scenario4(dataset):
    C = dataset["C"]
    P_realized = dataset["P_realized"]

    plasmid_main = np.einsum(
        "ijp,p->ij",
        P_realized,
        dataset["beta_P_true"],
    )

    # B has shape dC x dP.
    interaction = np.einsum(
        "ic,cp,ijp->ij",
        C,
        dataset["B_true"],
        P_realized,
    )

    delta_true = plasmid_main + interaction

    y0_true = (
        ALPHA_TRUE
        + C @ dataset["beta_C_true"]
    )

    yij_true = (
        y0_true[:, None]
        + delta_true
    )

    return {
        "y0_true": y0_true,
        "yij_true": yij_true,
        "delta_true": delta_true,
    }


def estimated_effects_realized(dataset, fit):
    C = dataset["C"]
    P_realized = dataset["P_realized"]

    (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    ) = unpack_beta(
        fit["beta_hat"]
    )

    y0_hat = (
        alpha_hat
        + C @ beta_C_hat
    )

    plasmid_main_hat = np.einsum(
        "ijp,p->ij",
        P_realized,
        beta_P_hat,
    )

    interaction_hat = np.einsum(
        "ic,cp,ijp->ij",
        C,
        B_hat,
        P_realized,
    )

    delta_hat = (
        plasmid_main_hat
        + interaction_hat
    )

    yij_hat = (
        y0_hat[:, None]
        + delta_hat
    )

    return {
        "y0_hat": y0_hat,
        "yij_hat": yij_hat,
        "delta_hat": delta_hat,
    }


def estimated_effects_fixed(dataset, fit):
    C = dataset["C"]
    P_fixed = dataset["P_fixed"]

    (
        alpha_hat,
        beta_C_hat,
        beta_P_hat,
        B_hat,
    ) = unpack_beta(
        fit["beta_hat"]
    )

    y0_hat = (
        alpha_hat
        + C @ beta_C_hat
    )

    delta_hat = (
        P_fixed @ beta_P_hat
    )[None, :] + (
        C
        @ B_hat
        @ P_fixed.T
    )

    yij_hat = (
        y0_hat[:, None]
        + delta_hat
    )

    return {
        "y0_hat": y0_hat,
        "yij_hat": yij_hat,
        "delta_hat": delta_hat,
    }


def basic_metrics(true_values, estimated_values):
    true_values = np.asarray(
        true_values,
        dtype=float,
    ).ravel()

    estimated_values = np.asarray(
        estimated_values,
        dtype=float,
    ).ravel()

    error = (
        estimated_values
        - true_values
    )

    nonzero = (
        np.abs(true_values)
        > 1e-12
    )

    if nonzero.any():
        sign_accuracy = np.mean(
            np.sign(
                estimated_values[nonzero]
            )
            == np.sign(
                true_values[nonzero]
            )
        )
    else:
        sign_accuracy = np.nan

    return {
        "bias": float(
            np.mean(error)
        ),
        "rmse": float(
            np.sqrt(
                np.mean(error ** 2)
            )
        ),
        "sign_accuracy": float(
            sign_accuracy
        ),
    }


def pairwise_delta_delta(delta_matrix):
    upper_i, upper_k = np.triu_indices(
        delta_matrix.shape[0],
        k=1,
    )

    dd = (
        delta_matrix[upper_i, :]
        - delta_matrix[upper_k, :]
    )

    return (
        dd,
        upper_i,
        upper_k,
    )


def evaluate_scenario4_fits(dataset, fit_realized, fit_fixed):
    truth = true_effects_scenario4(
        dataset
    )

    realized_est = estimated_effects_realized(
        dataset,
        fit_realized,
    )

    fixed_est = estimated_effects_fixed(
        dataset,
        fit_fixed,
    )

    dd_true, pair_i, pair_k = (
        pairwise_delta_delta(
            truth["delta_true"]
        )
    )

    dd_realized, _, _ = (
        pairwise_delta_delta(
            realized_est["delta_hat"]
        )
    )

    dd_fixed, _, _ = (
        pairwise_delta_delta(
            fixed_est["delta_hat"]
        )
    )

    delta_realized_metrics = basic_metrics(
        truth["delta_true"],
        realized_est["delta_hat"],
    )

    delta_fixed_metrics = basic_metrics(
        truth["delta_true"],
        fixed_est["delta_hat"],
    )

    dd_realized_metrics = basic_metrics(
        dd_true,
        dd_realized,
    )

    dd_fixed_metrics = basic_metrics(
        dd_true,
        dd_fixed,
    )

    yij_realized_metrics = basic_metrics(
        truth["yij_true"],
        realized_est["yij_hat"],
    )

    yij_fixed_metrics = basic_metrics(
        truth["yij_true"],
        fixed_est["yij_hat"],
    )

    row = {
        "delta_realized_bias":
            delta_realized_metrics["bias"],
        "delta_realized_rmse":
            delta_realized_metrics["rmse"],
        "delta_realized_sign_accuracy":
            delta_realized_metrics["sign_accuracy"],

        "delta_fixed_bias":
            delta_fixed_metrics["bias"],
        "delta_fixed_rmse":
            delta_fixed_metrics["rmse"],
        "delta_fixed_sign_accuracy":
            delta_fixed_metrics["sign_accuracy"],

        "delta_delta_realized_bias":
            dd_realized_metrics["bias"],
        "delta_delta_realized_rmse":
            dd_realized_metrics["rmse"],
        "delta_delta_realized_sign_accuracy":
            dd_realized_metrics["sign_accuracy"],

        "delta_delta_fixed_bias":
            dd_fixed_metrics["bias"],
        "delta_delta_fixed_rmse":
            dd_fixed_metrics["rmse"],
        "delta_delta_fixed_sign_accuracy":
            dd_fixed_metrics["sign_accuracy"],

        "yij_realized_rmse":
            yij_realized_metrics["rmse"],
        "yij_fixed_rmse":
            yij_fixed_metrics["rmse"],

        "sigma_g2_realized_hat":
            fit_realized["sigma_g2_hat"],
        "sigma_e2_realized_hat":
            fit_realized["sigma_e2_hat"],

        "sigma_g2_fixed_hat":
            fit_fixed["sigma_g2_hat"],
        "sigma_e2_fixed_hat":
            fit_fixed["sigma_e2_hat"],
    }

    return {
        "metrics": row,
        "truth": truth,
        "realized_est": realized_est,
        "fixed_est": fixed_est,
        "dd_true": dd_true,
        "dd_realized": dd_realized,
        "dd_fixed": dd_fixed,
        "pair_i": pair_i,
        "pair_k": pair_k,
    }


example_evaluation = evaluate_scenario4_fits(
    example,
    fit_realized,
    fit_fixed,
)

example_metrics = example_evaluation[
    "metrics"
]

print("=" * 90)
print("CELL 7 — EFFECT RECOVERY: REALIZED FEATURES VS FIXED FEATURES")
print("=" * 90)

display(
    pd.DataFrame(
        [example_metrics]
    ).T.rename(
        columns={0: "value"}
    )
)

print("\nPrimary Simulation 4 comparison:")
print(
    f"Delta RMSE, realized features: "
    f"{example_metrics['delta_realized_rmse']:.6f}"
)
print(
    f"Delta RMSE, fixed features:    "
    f"{example_metrics['delta_fixed_rmse']:.6f}"
)
print(
    f"DeltaDelta RMSE, realized features: "
    f"{example_metrics['delta_delta_realized_rmse']:.6f}"
)
print(
    f"DeltaDelta RMSE, fixed features:    "
    f"{example_metrics['delta_delta_fixed_rmse']:.6f}"
)

print("\nCell 7: PASS")


In [ ]:
#@title Cell 8 - Run 100 independent Simulation 4 replicates

def run_one_simulation4_replicate(replicate_index):
    seed = (
        MASTER_SEED
        + 3000
        + int(replicate_index)
    )

    dataset = simulate_scenario4_dataset(
        seed
    )

    static_realized = prepare_reml_static(
        dataset["X_realized"],
        dataset["pathogen_index"],
        dataset["K"],
    )

    fit_realized_r = fit_section2_reml_gls(
        dataset["X_realized"],
        dataset["y"],
        dataset["pathogen_index"],
        dataset["K"],
        static=static_realized,
    )

    static_fixed = prepare_reml_static(
        dataset["X_fixed"],
        dataset["pathogen_index"],
        dataset["K"],
    )

    fit_fixed_r = fit_section2_reml_gls(
        dataset["X_fixed"],
        dataset["y"],
        dataset["pathogen_index"],
        dataset["K"],
        static=static_fixed,
    )

    evaluation = evaluate_scenario4_fits(
        dataset,
        fit_realized_r,
        fit_fixed_r,
    )

    metrics = evaluation[
        "metrics"
    ].copy()

    metrics["replicate"] = int(
        replicate_index + 1
    )

    metrics["seed"] = int(seed)

    return metrics


replicate_rows = []

start_time = time.time()

for r in range(
    N_SIM_REPLICATES
):
    replicate_rows.append(
        run_one_simulation4_replicate(r)
    )

    if (
        (r + 1) % 10 == 0
        or r == 0
        or r + 1 == N_SIM_REPLICATES
    ):
        elapsed = (
            time.time()
            - start_time
        )

        print(
            f"Completed {r + 1:3d}/{N_SIM_REPLICATES} replicates "
            f"({elapsed:.1f} seconds elapsed)"
        )

replicate_results = pd.DataFrame(
    replicate_rows
)

REPLICATE_RESULTS_PATH = (
    OUTPUT_DIR
    / "04_scenario4_pathogen_dependent_feature_replicate_metrics.csv"
)

replicate_results.to_csv(
    REPLICATE_RESULTS_PATH,
    index=False,
)

print("\nSaved:")
print(REPLICATE_RESULTS_PATH)
print("\nCell 8: PASS")


In [ ]:
#@title Cell 9 - Summarize the 100-replicate Simulation 4 benchmark

PRIMARY_METRICS = [
    "delta_realized_bias",
    "delta_realized_rmse",
    "delta_realized_sign_accuracy",
    "delta_fixed_bias",
    "delta_fixed_rmse",
    "delta_fixed_sign_accuracy",
    "delta_delta_realized_bias",
    "delta_delta_realized_rmse",
    "delta_delta_realized_sign_accuracy",
    "delta_delta_fixed_bias",
    "delta_delta_fixed_rmse",
    "delta_delta_fixed_sign_accuracy",
]

SUPPORTING_METRICS = [
    "yij_realized_rmse",
    "yij_fixed_rmse",
    "sigma_g2_realized_hat",
    "sigma_e2_realized_hat",
    "sigma_g2_fixed_hat",
    "sigma_e2_fixed_hat",
]

summary_rows = []

for metric in (
    PRIMARY_METRICS
    + SUPPORTING_METRICS
):
    values = replicate_results[
        metric
    ].to_numpy(
        dtype=float
    )

    summary_rows.append({
        "metric": metric,
        "mean": float(
            np.nanmean(values)
        ),
        "sd": float(
            np.nanstd(
                values,
                ddof=1,
            )
        ),
        "median": float(
            np.nanmedian(values)
        ),
        "q025": float(
            np.nanquantile(
                values,
                0.025,
            )
        ),
        "q975": float(
            np.nanquantile(
                values,
                0.975,
            )
        ),
    })

scenario4_summary = pd.DataFrame(
    summary_rows
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "04_scenario4_pathogen_dependent_feature_summary.csv"
)

scenario4_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

print("=" * 90)
print("SCENARIO 4 — 100-REPLICATE SUMMARY")
print("=" * 90)

print("\nPrimary comparison:")
display(
    scenario4_summary[
        scenario4_summary[
            "metric"
        ].isin(
            PRIMARY_METRICS
        )
    ].reset_index(
        drop=True
    )
)

print("\nSupporting diagnostics:")
display(
    scenario4_summary[
        scenario4_summary[
            "metric"
        ].isin(
            SUPPORTING_METRICS
        )
    ].reset_index(
        drop=True
    )
)

delta_rmse_ratio = (
    replicate_results["delta_fixed_rmse"]
    / replicate_results["delta_realized_rmse"]
)

dd_rmse_ratio = (
    replicate_results["delta_delta_fixed_rmse"]
    / replicate_results["delta_delta_realized_rmse"]
)

print("\nDirect loss from incorrectly fixing pathogen-dependent features:")
print(
    f"Mean Delta RMSE ratio, fixed/realized:      "
    f"{delta_rmse_ratio.mean():.3f}"
)
print(
    f"Mean DeltaDelta RMSE ratio, fixed/realized: "
    f"{dd_rmse_ratio.mean():.3f}"
)

print("\nSaved:")
print(SUMMARY_PATH)
print("\nCell 9: PASS")


In [ ]:
#@title Cell 10 - Representative parametric bootstrap for both model specifications

def simulate_parametric_bootstrap_y(
    rng,
    X,
    pathogen_index,
    K,
    beta_hat,
    sigma_g2_hat,
    sigma_e2_hat,
):
    u_star = draw_correlated_host_effect(
        rng,
        K,
        sigma_g2_hat,
    )

    epsilon_star = rng.normal(
        0.0,
        np.sqrt(
            sigma_e2_hat
        ),
        size=X.shape[0],
    )

    return (
        X @ beta_hat
        + u_star[pathogen_index]
        + epsilon_star
    )


delta_true = example_evaluation[
    "truth"
]["delta_true"]

dd_true = example_evaluation[
    "dd_true"
]

bootstrap_rng = np.random.default_rng(
    MASTER_SEED + 900000
)

# Store Delta intervals for all 4,000 effects.
bootstrap_delta_realized = np.empty(
    (N_BOOTSTRAP, N_PATHOGENS * N_PLASMIDS),
    dtype=np.float32,
)

bootstrap_delta_fixed = np.empty(
    (N_BOOTSTRAP, N_PATHOGENS * N_PLASMIDS),
    dtype=np.float32,
)

# DeltaDelta has 398,000 effects; store float32 to keep memory moderate.
bootstrap_dd_realized = np.empty(
    (N_BOOTSTRAP, dd_true.size),
    dtype=np.float32,
)

bootstrap_dd_fixed = np.empty(
    (N_BOOTSTRAP, dd_true.size),
    dtype=np.float32,
)

start_time = time.time()

for b in range(N_BOOTSTRAP):
    # Realized-feature bootstrap.
    y_star_realized = simulate_parametric_bootstrap_y(
        bootstrap_rng,
        example["X_realized"],
        example["pathogen_index"],
        example["K"],
        fit_realized["beta_hat"],
        fit_realized["sigma_g2_hat"],
        fit_realized["sigma_e2_hat"],
    )

    fit_star_realized = fit_section2_reml_gls(
        example["X_realized"],
        y_star_realized,
        example["pathogen_index"],
        example["K"],
        static=static_realized,
    )

    effects_star_realized = estimated_effects_realized(
        example,
        fit_star_realized,
    )

    dd_star_realized, _, _ = pairwise_delta_delta(
        effects_star_realized["delta_hat"]
    )

    bootstrap_delta_realized[b, :] = (
        effects_star_realized["delta_hat"]
        .ravel()
        .astype(np.float32)
    )

    bootstrap_dd_realized[b, :] = (
        dd_star_realized
        .ravel()
        .astype(np.float32)
    )

    # Fixed-feature bootstrap.
    y_star_fixed = simulate_parametric_bootstrap_y(
        bootstrap_rng,
        example["X_fixed"],
        example["pathogen_index"],
        example["K"],
        fit_fixed["beta_hat"],
        fit_fixed["sigma_g2_hat"],
        fit_fixed["sigma_e2_hat"],
    )

    fit_star_fixed = fit_section2_reml_gls(
        example["X_fixed"],
        y_star_fixed,
        example["pathogen_index"],
        example["K"],
        static=static_fixed,
    )

    effects_star_fixed = estimated_effects_fixed(
        example,
        fit_star_fixed,
    )

    dd_star_fixed, _, _ = pairwise_delta_delta(
        effects_star_fixed["delta_hat"]
    )

    bootstrap_delta_fixed[b, :] = (
        effects_star_fixed["delta_hat"]
        .ravel()
        .astype(np.float32)
    )

    bootstrap_dd_fixed[b, :] = (
        dd_star_fixed
        .ravel()
        .astype(np.float32)
    )

    if (
        (b + 1) % 20 == 0
        or b == 0
        or b + 1 == N_BOOTSTRAP
    ):
        elapsed = (
            time.time()
            - start_time
        )

        print(
            f"Completed {b + 1:3d}/{N_BOOTSTRAP} paired bootstrap refits "
            f"({elapsed:.1f} seconds elapsed)"
        )


def interval_summary(
    bootstrap_values,
    truth_values,
    label,
):
    truth_values = np.asarray(
        truth_values,
        dtype=float,
    ).ravel()

    low = np.quantile(
        bootstrap_values,
        0.025,
        axis=0,
    )

    high = np.quantile(
        bootstrap_values,
        0.975,
        axis=0,
    )

    contains_truth = (
        (low <= truth_values)
        & (truth_values <= high)
    )

    excludes_zero = (
        (low > 0)
        | (high < 0)
    )

    return {
        "effect": label,
        "number_of_effects": int(
            len(truth_values)
        ),
        "fraction_CI_excludes_zero": float(
            np.mean(excludes_zero)
        ),
        "fraction_CI_contains_known_truth": float(
            np.mean(contains_truth)
        ),
        "mean_CI_width": float(
            np.mean(high - low)
        ),
    }


bootstrap_summary = pd.DataFrame([
    interval_summary(
        bootstrap_delta_realized,
        delta_true,
        "Delta_realized_feature_fit",
    ),
    interval_summary(
        bootstrap_delta_fixed,
        delta_true,
        "Delta_fixed_feature_fit",
    ),
    interval_summary(
        bootstrap_dd_realized,
        dd_true,
        "DeltaDelta_realized_feature_fit",
    ),
    interval_summary(
        bootstrap_dd_fixed,
        dd_true,
        "DeltaDelta_fixed_feature_fit",
    ),
])

BOOTSTRAP_SUMMARY_PATH = (
    OUTPUT_DIR
    / "04_scenario4_representative_bootstrap_summary.csv"
)

bootstrap_summary.to_csv(
    BOOTSTRAP_SUMMARY_PATH,
    index=False,
)

display(
    bootstrap_summary
)

print("\nImportant:")
print(
    "The fixed-feature model is intentionally misspecified: the data are generated "
    "from pathogen-specific copy-number and fitness values, but the fitted model "
    "uses only plasmid-level fixed values."
)

print("\nSaved:")
print(BOOTSTRAP_SUMMARY_PATH)
print("\nCell 10: PASS")


In [ ]:
#@title Cell 11 - Optional repeated-dataset bootstrap coverage

print(
    "Formal repeated-dataset bootstrap coverage remains optional and is "
    "disabled by default, as in Simulations 1-3."
)

if RUN_FULL_BOOTSTRAP_COVERAGE:
    print(
        "RUN_FULL_BOOTSTRAP_COVERAGE=True was requested, but the notebook "
        "does not automatically launch the very expensive repeated-dataset "
        "bootstrap calculation. Inspect Cell 10 first."
    )
else:
    print(
        "RUN_FULL_BOOTSTRAP_COVERAGE=False: no formal repeated-dataset "
        "coverage run was performed."
    )

print("\nCell 11: PASS")


In [ ]:
#@title Cell 12 - Final QC and output manifest

required_output_files = [
    REALIZED_FEATURE_PATH,
    REPLICATE_RESULTS_PATH,
    SUMMARY_PATH,
    BOOTSTRAP_SUMMARY_PATH,
]

missing_outputs = [
    str(path)
    for path in required_output_files
    if not path.exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "Required Simulation 4 output(s) are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )

manifest = {
    "notebook":
        "10_Simulation_04_Pathogen_Dependent_Plasmid_Features.ipynb",

    "scenario":
        "Simulation 04 - pathogen-dependent plasmid features",

    "central_question":
        "What is the consequence of treating pathogen-dependent plasmid "
        "features as invariant across pathogens?",

    "simulation1_biology_retained":
        True,

    "scenario4_change": {
        "pathogen_dependent_features": [
            "log2 TEM-1 copy number relative to empirical median",
            "plasmid-associated fitness score",
        ],

        "comparison": [
            "realized pathogen-specific features supplied to the fitted model",
            "same features incorrectly treated as fixed plasmid properties",
        ],

        "true_MIC_generated_from_realized_features_in_both_cases":
            True,
    },

    "scenario4_simulation_assumptions": {
        "fitness_beta":
            float(BETA_P_TRUE[5]),

        "baseline_fitness_SD":
            BASELINE_FITNESS_SD,

        "CN_host_shift_SD":
            CN_HOST_SHIFT_SD,

        "CN_pair_deviation_SD":
            CN_PAIR_DEVIATION_SD,

        "fitness_host_shift_SD":
            FITNESS_HOST_SHIFT_SD,

        "fitness_pair_deviation_SD":
            FITNESS_PAIR_DEVIATION_SD,
    },

    "simulation_settings": {
        "pathogens":
            N_PATHOGENS,

        "plasmids":
            N_PLASMIDS,

        "complete_observations":
            N_PATHOGENS
            * (N_PLASMIDS + 1),

        "alpha":
            ALPHA_TRUE,

        "sigma_g":
            SIGMA_G_TRUE,

        "sigma_e":
            SIGMA_E_TRUE,

        "simulation_replicates":
            N_SIM_REPLICATES,

        "bootstrap_replicates":
            N_BOOTSTRAP,
    },

    "evaluation_targets": [
        "Delta_ij recovery with realized pathogen-specific plasmid features",
        "Delta_ij recovery when pathogen-dependent plasmid features are incorrectly fixed",
        "DeltaDelta_ikj recovery under both feature assumptions",
    ],

    "outputs": [
        str(path)
        for path in required_output_files
    ],
}

MANIFEST_PATH = (
    OUTPUT_DIR
    / "04_scenario4_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

print("=" * 90)
print("SIMULATION 4 NOTEBOOK COMPLETE")
print("=" * 90)
print(f"Manifest: {MANIFEST_PATH}")
print(f"Required output files checked: {len(required_output_files)}")
print(
    "\nSimulation 4 compares correct pathogen-specific plasmid features "
    "against the incorrect assumption that those features are fixed."
)
print("Cell 12: PASS")
